In [1]:
# !pip install -q -U "transformers>=4.45" "trl>=0.11" "peft>=0.13" \
#     "datasets>=2.20" "accelerate>=0.34" "tokenizers>=0.20"

In [2]:
import torch, transformers, trl, peft, datasets

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # 'Qwen/Qwen2.5-0.5B-Instruct'

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [6]:
tokenizer.pad_token  # padding

'<|endoftext|>'

In [7]:
tokenizer.eos_token # end of sentence 

'<|im_end|>'

In [21]:
tokenizer.bos_token

In [ ]:
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# float64 : 64bit 구성된 _ _ _ _ _ _ _ _ .... _ _ _   12.34, 1.234, 123.4 -1000000000.00000000001     +1000000000.00000000001
# torch.float32 : precision                                                        -100000000.0000000001      +100000000.0000000001

# torch.float16 :                                                          -1000000.0000000001      +1000000.0000000001
#     _ |  _ _ _ _ _ _ _ _ |  _ _ _ _ _ _ _ _
    
# torch.bfloat16
#     _ |  _ _ _ _ _ _ _ _   _ _ _ _|  _ _ _ _                             -100000000.00001      +100000000.000001

In [9]:
model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            dtype = torch.bfloat16,
            device_map = 'auto'
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [10]:
sum(p.numel() for p in model.parameters())

1543714304

In [11]:
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('260519_ft')
TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH = DATA_DIR / 'val.jsonl'


In [13]:
ds = load_dataset('json', data_files = {'train' : str(TRAIN_PATH), 'validation' : str(VAL_PATH)})

In [15]:
ds['train'][0]['messages']

[{'role': 'system', 'content': '당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다.'},
 {'role': 'user', 'content': '결재 빨리 해.'},
 {'role': 'assistant', 'content': '결재 처리 가능 시점을 알려주실 수 있을까요?'}]

In [17]:
print(tokenizer.apply_chat_template(ds['train'][0]['messages'], tokenize=False, add_generation_prompt=False))

<|im_start|>system
당신은 무례한 문장을 정중한 이메일 문장으로 바꿔주는 어시스턴트입니다.<|im_end|>
<|im_start|>user
결재 빨리 해.<|im_end|>
<|im_start|>assistant
결재 처리 가능 시점을 알려주실 수 있을까요?<|im_end|>



In [22]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [26]:
MODEL_ID

'Qwen/Qwen2.5-1.5B-Instruct'

In [24]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [23]:
lora_cfg = LoraConfig(
        r = 16,
        lora_alpha=32,
        lora_dropout=0.05,  # 모든 파라미터를 다 학습 -> 학습이 너무 잘된다 overfitting
        bias = 'none',
        task_type = 'CAUSAL_LM',
        target_modules = [
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
        ]
)

In [27]:
model = get_peft_model(model, lora_cfg)

In [28]:
model.print_trainable_parameters() # freeze

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
